# 03 — Feature Engineering

**Layer:** Silver → Feature store  
**Objective:** Engineer time-series features and supervised targets for the three forecasting models, at the provider × specialty × month grain.  
**Input:** `data/interim/rtt_unified.parquet`.  
**Output:** `data/processed/rtt_features.parquet`.

**Series definition:** each `(provider_code, specialty_code)` pair is an independent monthly time series.

**Leakage policy:** features at month *t* use only information available up to and including *t* (lags, trailing rolling windows, current-month ratios). Targets are the *next* month's values (`shift(-1)` within series). The final month of each series therefore has undefined targets.

**Targets:**
- `target_total_next` — next month's waiting list size (demand regression).
- `target_pct_within_18_next` — next month's share treated within 18 weeks (waiting-time regression).
- `target_breach_next` — next month fails the 92% standard, i.e. `pct_within_18 < 0.92` (breach classification).

**Execution:** select the `Python (HCIP)` kernel, then run all cells in order.

## 1. Configuration and load

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def resolve_dir(*parts: str) -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base.joinpath(*parts)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate '{Path(*parts)}'.")


INPUT_PATH = resolve_dir("data", "interim") / "rtt_unified.parquet"
OUTPUT_PATH = resolve_dir("data", "processed") / "rtt_features.parquet"

SERIES = ["provider_code", "specialty_code"]
STATIC_COLS = ["icb_code", "icb_name", "provider_name", "specialty_name"]
STANDARD = 0.92  # 92% of pathways within 18 weeks

df = (
    pd.read_parquet(INPUT_PATH)
    .drop(columns=["Period"])
    .sort_values(SERIES + ["period_date"], ignore_index=True)
)
print(f"Loaded {len(df):,} rows | {df.groupby(SERIES).ngroups:,} series | "
      f"{df['period_date'].nunique()} periods")
df.head()

Loaded 86,571 rows | 4,161 series | 24 periods


,period_date,icb_code,icb_name,provider_code,provider_name,specialty_code,specialty_name,total_waiting,within_18wk,over_18wk,over_52wk,over_104wk,pct_within_18wk,breach_rate
0,2024-06-01,QOP,NHS GREATER MANCHESTER INTEGRATED CARE BOARD,8KL73,ENDOCARE DIAGNOSTICS,C_301,Gastroenterology Service,70.0,70.0,0.0,0.0,0.0,1.0,0.0
1,2024-07-01,QOP,NHS GREATER MANCHESTER INTEGRATED CARE BOARD,8KL73,ENDOCARE DIAGNOSTICS,C_301,Gastroenterology Service,55.0,55.0,0.0,0.0,0.0,1.0,0.0
2,2024-08-01,QOP,NHS GREATER MANCHESTER INTEGRATED CARE BOARD,8KL73,ENDOCARE DIAGNOSTICS,C_301,Gastroenterology Service,51.0,51.0,0.0,0.0,0.0,1.0,0.0
3,2024-04-01,QR1,NHS GLOUCESTERSHIRE INTEGRATED CARE BOARD,A0C5S,SPAMEDICA GLOUCESTER,C_130,Ophthalmology Service,61.0,61.0,0.0,0.0,0.0,1.0,0.0
4,2024-05-01,QR1,NHS GLOUCESTERSHIRE INTEGRATED CARE BOARD,A0C5S,SPAMEDICA GLOUCESTER,C_130,Ophthalmology Service,68.0,68.0,0.0,0.0,0.0,1.0,0.0


## 2. Regularise to continuous monthly series

Lag and shift operations assume consecutive monthly observations. Some series skip months; reindex each series to a complete month-start index over its own observed span, so every lag and target references the correct calendar month. Filled months carry missing measures (not zero) and are excluded from the supervised set downstream.

In [2]:
def regularise(frame: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for keys, g in frame.groupby(SERIES, sort=False):
        g = g.set_index("period_date").sort_index()
        full = pd.date_range(g.index.min(), g.index.max(), freq="MS")
        g = g.reindex(full)
        g[SERIES[0]], g[SERIES[1]] = keys
        g[STATIC_COLS] = g[STATIC_COLS].ffill().bfill()
        parts.append(g.rename_axis("period_date").reset_index())
    return pd.concat(parts, ignore_index=True)


rows_before = len(df)
gaps_before = int((df.groupby(SERIES)["period_date"].diff().dt.days > 31).sum())
df = regularise(df).sort_values(SERIES + ["period_date"], ignore_index=True)
gaps_after = int((df.groupby(SERIES)["period_date"].diff().dt.days > 31).sum())

print(f"Gaps (>1 month) before: {gaps_before:,}   after: {gaps_after:,}")
print(f"Rows: {rows_before:,} -> {len(df):,} (filled {len(df) - rows_before:,} missing months)")

Gaps (>1 month) before: 659   after: 0
Rows: 86,571 -> 88,678 (filled 2,107 missing months)


## 3. Calendar features

Month and quarter capture seasonality; the sine/cosine encoding represents month as a continuous cyclical signal (so December and January are adjacent).

In [3]:
df["month"] = df["period_date"].dt.month
df["quarter"] = df["period_date"].dt.quarter
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

## 4. Current-month ratio features

Long-waiter shares describe the shape of the waiting list at month *t*.

In [4]:
df["over_52_share"] = df["over_52wk"] / df["total_waiting"]
df["over_104_share"] = df["over_104wk"] / df["total_waiting"]

## 5. Lag features

Prior-month values of the waiting list and breach rate, computed within each series.

In [5]:
grp = df.groupby(SERIES)
for k in (1, 2, 3, 12):
    df[f"lag{k}_total"] = grp["total_waiting"].shift(k)
df["lag1_breach"] = grp["breach_rate"].shift(1)
df["lag12_breach"] = grp["breach_rate"].shift(12)

## 6. Rolling-window features

Trailing means and volatility of the waiting list, within each series.

In [6]:
for w in (3, 6, 12):
    df[f"roll{w}_total"] = grp["total_waiting"].transform(
        lambda s, w=w: s.rolling(w, min_periods=1).mean()
    )
df["roll3_std_total"] = grp["total_waiting"].transform(
    lambda s: s.rolling(3, min_periods=2).std()
)

## 7. Growth features

Month-on-month and year-on-year change in the waiting list.

In [7]:
df["mom_change_total"] = df["total_waiting"] - df["lag1_total"]
df["mom_pct_total"] = df["total_waiting"] / df["lag1_total"] - 1
df["yoy_pct_total"] = df["total_waiting"] / df["lag12_total"] - 1
df[["mom_pct_total", "yoy_pct_total"]] = df[["mom_pct_total", "yoy_pct_total"]].replace(
    [np.inf, -np.inf], np.nan
)

## 8. Targets (next month)

Shift target columns back by one month within each series. The final month of every series has undefined targets and is excluded from the supervised set.

In [8]:
df["target_total_next"] = grp["total_waiting"].shift(-1)
df["target_pct_within_18_next"] = grp["pct_within_18wk"].shift(-1)
df["target_breach_next"] = (df["target_pct_within_18_next"] < STANDARD).astype("Int8")
df.loc[df["target_pct_within_18_next"].isna(), "target_breach_next"] = pd.NA

## 9. Validation and persist

Confirm features reference the prior month and targets reference the next month, then write the feature table.

In [9]:
feature_cols = [
    "month", "quarter", "month_sin", "month_cos",
    "over_52_share", "over_104_share",
    "lag1_total", "lag2_total", "lag3_total", "lag12_total",
    "lag1_breach", "lag12_breach",
    "roll3_total", "roll6_total", "roll12_total", "roll3_std_total",
    "mom_change_total", "mom_pct_total", "yoy_pct_total",
]
target_cols = ["target_total_next", "target_pct_within_18_next", "target_breach_next"]

# Temporal-alignment check on the longest series.
sample_key = df.groupby(SERIES).size().idxmax()
s = df[(df["provider_code"] == sample_key[0]) & (df["specialty_code"] == sample_key[1])]
leak_ok = bool((s["lag1_total"].iloc[1:].values == s["total_waiting"].iloc[:-1].values).all())
target_ok = bool((s["target_total_next"].iloc[:-1].values == s["total_waiting"].iloc[1:].values).all())

print(f"lag1 aligns to previous month : {leak_ok}")
print(f"target aligns to next month   : {target_ok}")
assert leak_ok and target_ok, "Temporal alignment check failed."

supervised = df.dropna(subset=target_cols + ["lag1_total"]).reset_index(drop=True)
print(f"\nTotal rows (incl. filled months): {len(df):,}")
print(f"Supervised rows (usable):         {len(supervised):,}")
print(f"Feature columns:                  {len(feature_cols)}")
print(f"Breach class balance:             {supervised['target_breach_next'].mean():.3f} positive")

df.to_parquet(OUTPUT_PATH, index=False)
print(f"\nWritten: {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size / 1_000:.0f} KB)")

lag1 aligns to previous month : True
target aligns to next month   : True

Total rows (incl. filled months): 88,678
Supervised rows (usable):         77,651
Feature columns:                  19
Breach class balance:             0.846 positive

Written: f:\healthcare-capacity-intelligence-platform\data\processed\rtt_features.parquet (7294 KB)


In [10]:
df[SERIES + ["period_date", "total_waiting", "lag1_total", "roll3_total", "yoy_pct_total", "target_total_next", "target_breach_next"]].head(15)

,provider_code,specialty_code,period_date,total_waiting,lag1_total,roll3_total,yoy_pct_total,target_total_next,target_breach_next
0,8KL73,C_301,2024-06-01,70.0,NaN,70.000000,NaN,55.0,0
1,8KL73,C_301,2024-07-01,55.0,70.0,62.500000,NaN,51.0,0
2,8KL73,C_301,2024-08-01,51.0,55.0,58.666667,NaN,NaN,<NA>
3,A0C5S,C_130,2024-04-01,61.0,NaN,61.000000,NaN,68.0,0
4,A0C5S,C_130,2024-05-01,68.0,61.0,64.500000,NaN,91.0,0
5,A0C5S,C_130,2024-06-01,91.0,68.0,73.333333,NaN,82.0,0
6,A0C5S,C_130,2024-07-01,82.0,91.0,80.333333,NaN,88.0,0
7,A0C5S,C_130,2024-08-01,88.0,82.0,87.000000,NaN,78.0,0
8,A0C5S,C_130,2024-09-01,78.0,88.0,82.666667,NaN,97.0,0
9,A0C5S,C_130,2024-10-01,97.0,78.0,87.666667,NaN,107.0,0
